In [ ]:
import sys
import pandas as pd
import numpy as np
import os
import importlib
from src.eval_utils import full_evaluation
from src.modeltrainer import *
import matplotlib.pyplot as plt
import glob

# Reimporter les modules pour refléter les modifications
importlib.reload(sys.modules['src.eval_utils'])
importlib.reload(sys.modules['src.modeltrainer'])

sys.path.insert(0, './src')

In [ ]:
# =============================================================================
# CELLULE 1 : IMPORT DES DONNÉES
# =============================================================================


# Configuration
DATA_PATH = 'data/processed_data_imputed.xlsx'
COLS_TO_DROP = ['patient_ids', 'feat_40', 'feat_82']

# Chargement des données
print("📂 Chargement des données...")

train_data = pd.read_excel(DATA_PATH, sheet_name='train')
cal_data = pd.read_excel(DATA_PATH, sheet_name='calibration')
test_data = pd.read_excel(DATA_PATH, sheet_name='test')

# Séparation features / target
X_train = train_data.iloc[:, :-1].drop(columns=COLS_TO_DROP, errors='ignore')
y_train = train_data.iloc[:, -1]

X_cal = cal_data.iloc[:, :-1].drop(columns=COLS_TO_DROP, errors='ignore')
y_cal = cal_data.iloc[:, -1]

X_test = test_data.iloc[:, :-1].drop(columns=COLS_TO_DROP, errors='ignore')
y_test = test_data.iloc[:, -1]

print(f"✓ Données chargées:")
print(f"   - Train: {X_train.shape[0]} lignes, {X_train.shape[1]} features")
print(f"   - Calibration: {X_cal.shape[0]} lignes")
print(f"   - Test: {X_test.shape[0]} lignes")
print(f"   - Target distribution (train): {y_train.value_counts().to_dict()}")

In [ ]:
# =============================================================================
# CELLULE 2 : CHARGEMENT OU ENTRAÎNEMENT DU MODÈLE RandomForest
# =============================================================================


# Configuration
MODEL_DIR = 'models'
MODEL_TYPE = 'RandomForest'  # 'LogisticRegression', 'RandomForest', 'XGBoost'



# Initialiser le trainer
trainer = ModelML(
    x_train=X_train,
    x_test=X_test,
    y_train=y_train,
    y_test=y_test
)

# Chercher un modèle existant
existing_model = find_existing_model(MODEL_DIR, MODEL_TYPE)

if existing_model:
    print(f"📦 Modèle existant trouvé: {existing_model}")
    trainer.load_model(existing_model)
else:
    print(f"🔄 Aucun modèle {MODEL_TYPE} trouvé. Entraînement en cours...")
    
    trainer.tune_random_forest(
        param_grid={
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 10, None],
            'min_samples_leaf': [1, 2],
            'max_features': ['sqrt']
        },
        method='random',
        n_iter=20,
        cv=5,
        scoring='roc_auc'
    )
    
    trainer.save_model()
    print(f"✓ Modèle entraîné et sauvegardé")

print(f"\n📊 Modèle actif: {trainer.model_name}")
print(f"   Paramètres: {trainer.best_params}")

In [ ]:
# =============================================================================
# CELLULE 3 : ÉVALUATION COMPLÈTE SUR LE TRAIN ET TEST SET
# =============================================================================

# --- Évaluation sur le TRAIN SET ---
print("\n" + "="*70)
print("📊 ÉVALUATION SUR LE TRAIN SET")
print("="*70)

y_pred_train = trainer.predict(X_train)
y_proba_train = trainer.predict_proba(X_train)[:, 1]

metrics_train = full_evaluation(
    y_true=y_train,
    y_pred=y_pred_train,
    y_proba=y_proba_train,
    model_name=f"{trainer.model_name}_train",
    save_dir='results'
)

# --- Évaluation sur le TEST SET ---
print("\n" + "="*70)
print("📊 ÉVALUATION SUR LE TEST SET")
print("="*70)

y_pred_test = trainer.predict(X_test)
y_proba_test = trainer.predict_proba(X_test)[:, 1]

metrics_test = full_evaluation(
    y_true=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=f"{trainer.model_name}_test",
    save_dir='results'
)

# Afficher les plots
plt.show()

# Résumé comparatif
print("\n" + "="*70)
print("📈 COMPARAISON TRAIN vs TEST")
print("="*70)
print(f"{'Métrique':<25} {'Train':>12} {'Test':>12} {'Diff':>12}")
print("-"*70)
for key in metrics_train.keys():
    train_val = metrics_train[key]
    test_val = metrics_test[key]
    diff = train_val - test_val
    print(f"{key:<25} {train_val:>12.4f} {test_val:>12.4f} {diff:>+12.4f}")